# Main Decoding Methods in NLP
## Comprehensive Guide with Working Examples, Advantages, Disadvantages & Use Cases

Decoding methods are strategies used in sequence-to-sequence models to generate text from a language model. They determine how the next token is selected at each step of generation.

In [ ]:
# Install required packages
# !pip install transformers torch numpy pandas matplotlib -q

## 1. Greedy Search
### Purpose: Select the token with highest probability at each step

At each generation step, always pick the token with the maximum probability, without considering future tokens.

In [ ]:
import numpy as np
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel

# Manual Greedy Search Implementation
class GreedySearchDecoder:
    def __init__(self, vocab_size=50257):  # GPT2 vocab size
        self.vocab_size = vocab_size
    
    def decode(self, logits, max_length=10):
        """
        Greedy Search: always select token with highest probability
        """
        sequence = []
        
        for step in range(max_length):
            # Get probabilities from logits
            probabilities = np.exp(logits[step]) / np.sum(np.exp(logits[step]))
            # Select token with maximum probability
            next_token = np.argmax(probabilities)
            sequence.append(next_token)
            
        return sequence

# Example logits (simulate model output)
np.random.seed(42)
example_logits = np.random.randn(5, 10)  # 5 time steps, 10 vocab size

decoder = GreedySearchDecoder(vocab_size=10)
greedy_sequence = decoder.decode(example_logits, max_length=5)

print("Greedy Search Example:")
print(f"Selected tokens: {greedy_sequence}")
print(f"\nStep-by-step selection:")
for step in range(5):
    probs = np.exp(example_logits[step]) / np.sum(np.exp(example_logits[step]))
    max_token = np.argmax(probs)
    max_prob = np.max(probs)
    print(f"Step {step}: Token={max_token}, Probability={max_prob:.4f}")

In [ ]:
# Using Hugging Face Transformers
from transformers import GPT2Tokenizer, GPT2LMHeadModel

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

# Greedy Search decoding
prompt = "The quick brown fox"
input_ids = tokenizer.encode(prompt, return_tensors="pt")

# Generate with greedy search
output_greedy = model.generate(
    input_ids,
    max_length=20,
    do_sample=False,  # Deterministic (greedy)
    num_beams=1,      # Single hypothesis (greedy)
    temperature=1.0
)

greedy_text = tokenizer.decode(output_greedy[0], skip_special_tokens=True)
print(f"\nGreedy Search Output:")
print(f"Input: {prompt}")
print(f"Generated: {greedy_text}")

### Advantages:
- ✓ **Fast** - Single token selection per step, O(1) complexity
- ✓ **Simple** - Easy to implement and understand
- ✓ **Memory efficient** - No need to store multiple hypotheses
- ✓ **Deterministic** - Always produces same output for same input

### Disadvantages:
- ✗ **Suboptimal** - May get stuck in locally optimal but globally suboptimal sequences
- ✗ **Repetition** - Prone to generating repetitive text
- ✗ **Limited diversity** - No variations in generated text
- ✗ **Cannot backtrack** - One wrong early token ruins entire sequence
- ✗ **Poor quality** - Often produces incoherent sequences

### Use Cases:
- Fast/simple text generation for real-time applications
- Machine translation (as baseline)
- Summarization (fast generation)
- Initial prototyping
- Resource-constrained environments
- Quick demonstrations

## 2. Beam Search
### Purpose: Keep top-k hypotheses at each step and explore multiple paths

Maintains the k most likely sequences (beams) at each step, balancing quality and diversity.

In [ ]:
# Manual Beam Search Implementation
class BeamSearchDecoder:
    def __init__(self, beam_width=3, vocab_size=10):
        self.beam_width = beam_width
        self.vocab_size = vocab_size
    
    def decode(self, logits, max_length=5):
        """
        Beam Search: maintain top-k hypotheses
        Returns best sequence and all beam sequences
        """
        # Initialize with empty sequence
        beams = [([], 0.0)]  # (sequence, log_probability)
        
        for step in range(max_length):
            all_candidates = []
            
            for seq, log_prob in beams:
                # Get probabilities from logits at current step
                logits_step = logits[step]
                log_probs = logits_step - np.log(np.sum(np.exp(logits_step)))
                
                # Generate candidates for this beam
                for token_id in range(self.vocab_size):
                    new_seq = seq + [token_id]
                    new_log_prob = log_prob + log_probs[token_id]
                    all_candidates.append((new_seq, new_log_prob))
            
            # Select top-k candidates
            all_candidates.sort(key=lambda x: x[1], reverse=True)
            beams = all_candidates[:self.beam_width]
        
        return beams

# Example with synthetic logits
np.random.seed(42)
example_logits = np.random.randn(4, 10)  # 4 steps, 10 vocab

beam_decoder = BeamSearchDecoder(beam_width=3, vocab_size=10)
beam_results = beam_decoder.decode(example_logits, max_length=4)

print("Beam Search Results (beam_width=3):")
for i, (seq, prob) in enumerate(beam_results):
    print(f"Beam {i+1}: {seq}, Log-Prob: {prob:.4f}")

In [ ]:
# Using Hugging Face Transformers with Beam Search
prompt = "The quick brown fox"
input_ids = tokenizer.encode(prompt, return_tensors="pt")

# Generate with beam search
output_beam = model.generate(
    input_ids,
    max_length=20,
    num_beams=5,           # Number of beams
    do_sample=False,       # Greedy selection within beam
    num_return_sequences=3, # Return top-3 sequences
    early_stopping=True
)

print("\nBeam Search (num_beams=5) Output:")
print(f"Input: {prompt}\n")
for i, output_id in enumerate(output_beam):
    beam_text = tokenizer.decode(output_id, skip_special_tokens=True)
    print(f"Beam {i+1}: {beam_text}")

### Advantages:
- ✓ **Better quality** - Explores multiple hypotheses
- ✓ **Less repetition** - Multiple beams reduce local optima
- ✓ **Controllable** - Beam width parameter controls depth vs speed
- ✓ **Proven effective** - Works well for translation, summarization
- ✓ **Diverse outputs** - Can generate multiple valid sequences

### Disadvantages:
- ✗ **Slower** - O(k*vocab_size) per step, k=beam width
- ✗ **Memory intensive** - Must store k sequences
- ✗ **Repetition still occurs** - Suffers from beam search curse
- ✗ **Limited diversity** - Beams tend to converge to similar sequences
- ✗ **Tuning required** - Beam width affects speed/quality tradeoff

### Use Cases:
- Machine translation (standard approach)
- Summarization with high quality requirement
- Title/headline generation
- Question answering systems
- Image captioning
- Code generation

## 3. Top-k Sampling
### Purpose: Sample from top-k most likely tokens

Filter to top-k most probable tokens and sample uniformly or according to probabilities.

In [ ]:
# Manual Top-k Sampling Implementation
class TopKSamplingDecoder:
    def __init__(self, k=5, vocab_size=100):
        self.k = k
        self.vocab_size = vocab_size
    
    def decode(self, logits, max_length=10):
        """
        Top-k Sampling: sample from top-k tokens
        """
        sequence = []
        
        for step in range(max_length):
            # Convert logits to probabilities
            logits_step = logits[step]
            probabilities = np.exp(logits_step) / np.sum(np.exp(logits_step))
            
            # Get top-k token indices and probabilities
            top_k_indices = np.argsort(probabilities)[-self.k:]
            top_k_probs = probabilities[top_k_indices]
            
            # Renormalize to sum to 1
            top_k_probs = top_k_probs / np.sum(top_k_probs)
            
            # Sample from top-k
            next_token = np.random.choice(top_k_indices, p=top_k_probs)
            sequence.append(next_token)
        
        return sequence

# Visualization of top-k filtering
np.random.seed(42)
example_logits = np.random.randn(100)  # 100 vocabulary size
probabilities = np.exp(example_logits) / np.sum(np.exp(example_logits))

# Top-5 tokens
top_5_indices = np.argsort(probabilities)[-5:]
top_5_probs = probabilities[top_5_indices]

print("Top-5 Sampling Example:")
print(f"Vocab size: {len(probabilities)}")
print(f"\nTop-5 token probabilities:")
for idx, token_id in enumerate(sorted(top_5_indices)):
    print(f"Token {token_id}: {probabilities[token_id]:.4f}")

# Multiple samples
decoder = TopKSamplingDecoder(k=5, vocab_size=100)
print(f"\n3 Samples from Top-5:")
for i in range(3):
    sample = decoder.decode(np.array([example_logits, example_logits]), max_length=2)
    print(f"Sample {i+1}: {sample}")

In [ ]:
# Using Hugging Face Transformers with Top-k Sampling
prompt = "In the beginning"
input_ids = tokenizer.encode(prompt, return_tensors="pt")

# Generate with top-k sampling
output_top_k = model.generate(
    input_ids,
    max_length=20,
    do_sample=True,        # Enable sampling
    top_k=50,              # Top-50 tokens
    temperature=0.7,
    num_return_sequences=3  # 3 different samples
)

print("\nTop-k Sampling (k=50) Output:")
print(f"Input: {prompt}\n")
for i, output_id in enumerate(output_top_k):
    text = tokenizer.decode(output_id, skip_special_tokens=True)
    print(f"Sample {i+1}: {text}")

### Advantages:
- ✓ **Diverse** - Multiple different outputs from same input
- ✓ **Creative** - Good for creative text generation
- ✓ **Fast** - Faster than beam search
- ✓ **Practical** - Works well in practice
- ✓ **Stochastic** - Adds natural randomness

### Disadvantages:
- ✗ **Inconsistent quality** - Some samples may be low quality
- ✗ **Hard to control** - k is arbitrary parameter
- ✗ **May exclude good tokens** - k too small eliminates valid options
- ✗ **Ignores tail probability** - Truncates valid but low-prob tokens
- ✗ **Not reproducible** - Stochastic process

### Use Cases:
- Creative text generation (stories, poetry)
- Dialogue systems (diverse responses)
- Paraphrase generation
- Data augmentation
- Conversational AI
- Open-ended generation

## 4. Top-p (Nucleus) Sampling
### Purpose: Sample from smallest set of tokens with cumulative probability ≥ p

More adaptive than top-k: uses cumulative probability instead of fixed token count.

In [ ]:
# Manual Top-p (Nucleus) Sampling Implementation
class TopPSamplingDecoder:
    def __init__(self, p=0.9, vocab_size=100):
        self.p = p
        self.vocab_size = vocab_size
    
    def decode(self, logits, max_length=10):
        """
        Top-p (Nucleus) Sampling: sample from tokens with cumulative prob >= p
        """
        sequence = []
        
        for step in range(max_length):
            logits_step = logits[step]
            probabilities = np.exp(logits_step) / np.sum(np.exp(logits_step))
            
            # Sort probabilities in descending order
            sorted_indices = np.argsort(probabilities)[::-1]
            sorted_probs = probabilities[sorted_indices]
            
            # Calculate cumulative probabilities
            cumsum_probs = np.cumsum(sorted_probs)
            
            # Find cutoff index where cumsum >= p
            cutoff_idx = np.where(cumsum_probs >= self.p)[0][0]
            nucleus_indices = sorted_indices[:cutoff_idx + 1]
            nucleus_probs = sorted_probs[:cutoff_idx + 1]
            
            # Renormalize
            nucleus_probs = nucleus_probs / np.sum(nucleus_probs)
            
            # Sample from nucleus
            next_token = np.random.choice(nucleus_indices, p=nucleus_probs)
            sequence.append(next_token)
        
        return sequence

# Visualization of nucleus sampling
np.random.seed(42)
example_logits = np.random.randn(100)
probabilities = np.exp(example_logits) / np.sum(np.exp(example_logits))

# Show different p values
for p_value in [0.5, 0.7, 0.9, 0.95]:
    sorted_indices = np.argsort(probabilities)[::-1]
    sorted_probs = probabilities[sorted_indices]
    cumsum_probs = np.cumsum(sorted_probs)
    cutoff_idx = np.where(cumsum_probs >= p_value)[0][0]
    nucleus_size = cutoff_idx + 1
    nucleus_prob = cumsum_probs[cutoff_idx]
    print(f"p={p_value}: nucleus_size={nucleus_size}, cumulative_prob={nucleus_prob:.4f}")

In [ ]:
# Comparison: Top-k vs Top-p
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Example probability distribution
np.random.seed(42)
logits = np.random.randn(50)
probs = np.exp(logits) / np.sum(np.exp(logits))
sorted_indices = np.argsort(probs)[::-1]
sorted_probs = probs[sorted_indices]

# Top-5
ax1.bar(range(len(sorted_probs[:15])), sorted_probs[:15], color='steelblue')
ax1.axvline(x=4.5, color='red', linestyle='--', label='Top-5 cutoff')
ax1.set_title('Top-k Sampling (k=5)')
ax1.set_xlabel('Token Rank')
ax1.set_ylabel('Probability')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Top-p (nucleus)
p_value = 0.9
cumsum_probs = np.cumsum(sorted_probs)
cutoff_idx = np.where(cumsum_probs >= p_value)[0][0]
colors = ['steelblue' if i <= cutoff_idx else 'lightgray' for i in range(len(sorted_probs[:15]))]
ax2.bar(range(len(sorted_probs[:15])), sorted_probs[:15], color=colors)
ax2.axvline(x=cutoff_idx + 0.5, color='red', linestyle='--', label=f'Top-p (p=0.9) cutoff')
ax2.set_title(f'Top-p (Nucleus) Sampling (p={p_value})')
ax2.set_xlabel('Token Rank')
ax2.set_ylabel('Probability')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nTop-5: selects first 5 tokens regardless of their probabilities")
print(f"Top-p (p=0.9): selects {cutoff_idx + 1} tokens to achieve 90% cumulative probability")

In [ ]:
# Using Hugging Face Transformers with Top-p Sampling
prompt = "The future of AI is"
input_ids = tokenizer.encode(prompt, return_tensors="pt")

# Generate with top-p sampling
output_top_p = model.generate(
    input_ids,
    max_length=20,
    do_sample=True,
    top_p=0.9,             # Nucleus (90% cumulative probability)
    temperature=0.7,
    num_return_sequences=3
)

print("\nTop-p Sampling (p=0.9) Output:")
print(f"Input: {prompt}\n")
for i, output_id in enumerate(output_top_p):
    text = tokenizer.decode(output_id, skip_special_tokens=True)
    print(f"Sample {i+1}: {text}")

### Advantages:
- ✓ **Adaptive** - Automatically adjusts to token probability distribution
- ✓ **Diverse** - Generates varied outputs
- ✓ **Theoretically sound** - Probability-based filtering
- ✓ **Better than top-k** - Handles varying distributions well
- ✓ **Interpretable** - p represents confidence threshold

### Disadvantages:
- ✗ **Stochastic** - Non-deterministic output
- ✗ **Parameter tuning** - Choosing p value requires experimentation
- ✗ **Still excludes tokens** - May miss valid low-probability tokens
- ✗ **Quality variance** - Some outputs may be lower quality
- ✗ **Complexity** - More complex than greedy or top-k

### Use Cases:
- High-quality creative generation (stories, poetry)
- Chat/conversational AI (natural diverse responses)
- Content generation (articles, summaries)
- Text completion
- Open-ended dialogue
- Modern language models (GPT-2, GPT-3 recommended method)

## 5. Temperature Sampling
### Purpose: Control randomness/diversity of generation

Scales logits before softmax: higher temperature = more random, lower = more deterministic.

In [ ]:
# Manual Temperature Sampling Implementation
class TemperatureSamplingDecoder:
    def __init__(self, temperature=1.0, vocab_size=100):
        self.temperature = temperature
        self.vocab_size = vocab_size
    
    def decode(self, logits, max_length=10):
        """
        Temperature Sampling:
        T < 1.0: more deterministic (sharper distribution)
        T = 1.0: unchanged
        T > 1.0: more random (flatter distribution)
        """
        sequence = []
        
        for step in range(max_length):
            logits_step = logits[step]
            
            # Apply temperature scaling
            scaled_logits = logits_step / self.temperature
            
            # Convert to probabilities
            probabilities = np.exp(scaled_logits) / np.sum(np.exp(scaled_logits))
            
            # Sample from distribution
            next_token = np.random.choice(len(probabilities), p=probabilities)
            sequence.append(next_token)
        
        return sequence

# Visualization of temperature effects
np.random.seed(42)
example_logits = np.array([1.0, 2.0, 0.5, 1.5, 0.8, 1.2, 0.3, 1.8, 0.9, 1.1])

temperatures = [0.3, 0.7, 1.0, 1.5, 2.0]

fig, ax = plt.subplots(figsize=(12, 5))

x = np.arange(len(example_logits))
width = 0.15

for i, temp in enumerate(temperatures):
    scaled_logits = example_logits / temp
    probs = np.exp(scaled_logits) / np.sum(np.exp(scaled_logits))
    offset = width * (i - len(temperatures) // 2)
    ax.bar(x + offset, probs, width, label=f'T={temp}')

ax.set_xlabel('Token ID')
ax.set_ylabel('Probability')
ax.set_title('Effect of Temperature on Probability Distribution')
ax.set_xticks(x)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Temperature Effects:")
print("T < 1.0 (e.g., 0.3): Sharp distribution (deterministic, less random)")
print("T = 1.0: Original distribution")
print("T > 1.0 (e.g., 2.0): Flat distribution (stochastic, more random)")

In [ ]:
# Using Hugging Face Transformers with different temperatures
prompt = "Machine learning is"
input_ids = tokenizer.encode(prompt, return_tensors="pt")

temperatures = [0.3, 0.7, 1.0, 1.5]

print("Temperature Sampling Comparison:\n")
for temp in temperatures:
    output = model.generate(
        input_ids,
        max_length=18,
        do_sample=True,
        temperature=temp,
        top_p=0.95
    )
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f"Temperature {temp}: {text}")

### Advantages:
- ✓ **Flexible** - Single parameter controls all diversity
- ✓ **Intuitive** - Easy to understand conceptually
- ✓ **Fast** - No additional computation
- ✓ **Combatable** - Works with other methods (top-k, top-p)
- ✓ **Effective** - Simple yet powerful

### Disadvantages:
- ✗ **Arbitrary** - No principled way to choose temperature value
- ✗ **Task dependent** - Requires tuning per task
- ✗ **Extreme values** - Very low/high temperatures can degrade quality
- ✗ **Global effect** - Affects all tokens equally
- ✗ **Can amplify errors** - High T may amplify model errors

### Use Cases:
- Fine-tuning diversity/creativity
- Combination with other sampling methods
- Controlling model confidence
- Task-specific adaptation
- Real-time generation control
- Interactive text generation systems

## 6. Contrastive Search
### Purpose: Balance model confidence with diversity (penalize repetition)

Selects tokens that are likely according to the model AND dissimilar to previously generated tokens.

In [ ]:
# Manual Contrastive Search Implementation
class ContrastiveSearchDecoder:
    def __init__(self, alpha=0.6, k=4, vocab_size=100):
        """
        alpha: weight for diversity penalty (0-1)
        k: number of top tokens to consider
        """
        self.alpha = alpha
        self.k = k
        self.vocab_size = vocab_size
    
    def cosine_similarity(self, u, v):
        """Calculate cosine similarity between two vectors"""
        return np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v) + 1e-8)
    
    def decode(self, logits, embeddings, max_length=10):
        """
        Contrastive Search:
        Score = model_score - alpha * max_diversity_penalty
        model_score: log probability from model
        diversity_penalty: similarity to previous tokens
        """
        sequence = []
        
        for step in range(max_length):
            logits_step = logits[step]
            
            # Get model scores (log probabilities)
            log_probs = logits_step - np.log(np.sum(np.exp(logits_step)))
            
            # Get top-k candidates
            top_k_indices = np.argsort(log_probs)[-self.k:]
            
            contrastive_scores = []
            
            for token_id in top_k_indices:
                # Model confidence
                model_score = log_probs[token_id]
                
                # Diversity penalty: max similarity to previous tokens
                diversity_penalty = 0
                if len(sequence) > 0:
                    prev_embeddings = [embeddings[prev_token] for prev_token in sequence[-3:]]  # Last 3 tokens
                    curr_embedding = embeddings[token_id]
                    
                    max_similarity = max([self.cosine_similarity(curr_embedding, prev_emb) 
                                         for prev_emb in prev_embeddings])
                    diversity_penalty = max_similarity
                
                # Contrastive score
                score = model_score - self.alpha * diversity_penalty
                contrastive_scores.append(score)
            
            # Select token with best contrastive score
            best_idx = np.argmax(contrastive_scores)
            next_token = top_k_indices[best_idx]
            sequence.append(next_token)
        
        return sequence

# Simulate embeddings (normally from model)
np.random.seed(42)
vocab_size = 100
embedding_dim = 50
embeddings = np.random.randn(vocab_size, embedding_dim)
embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)  # Normalize

# Example logits
example_logits = np.random.randn(5, vocab_size)

decoder = ContrastiveSearchDecoder(alpha=0.6, k=4)
sequence = decoder.decode(example_logits, embeddings, max_length=5)

print("Contrastive Search Example:")
print(f"Generated sequence: {sequence}")
print(f"\nAlpha={decoder.alpha} (diversity weight): higher = more diverse, lower = more confident")

In [ ]:
# Using Hugging Face Transformers with Contrastive Search
# Note: Contrastive search is available in newer versions of transformers

prompt = "Natural language processing has"
input_ids = tokenizer.encode(prompt, return_tensors="pt")

try:
    # Contrastive search (requires recent transformers version)
    output_contrastive = model.generate(
        input_ids,
        max_length=20,
        penalty_alpha=0.6,     # Diversity penalty weight
        top_k=4                # Top-k candidates
    )
    
    contrastive_text = tokenizer.decode(output_contrastive[0], skip_special_tokens=True)
    print("Contrastive Search Output:")
    print(f"Input: {prompt}")
    print(f"Generated: {contrastive_text}")
except Exception as e:
    print(f"Note: {e}")
    print("Contrastive search may require newer version of transformers library")

### Advantages:
- ✓ **Anti-repetition** - Explicitly penalizes repeated tokens
- ✓ **Balanced** - Maintains model confidence while increasing diversity
- ✓ **Coherent** - Generates more coherent text than sampling
- ✓ **Controllable** - Alpha parameter controls diversity-quality tradeoff
- ✓ **Recent** - Preferred method in recent literature

### Disadvantages:
- ✗ **Computationally expensive** - Requires embedding similarity calculations
- ✗ **Embedding dependent** - Quality depends on embedding space
- ✗ **Parameter tuning** - Requires tuning alpha and k
- ✗ **Memory intensive** - Must store embeddings for all tokens
- ✗ **Complex** - More complex than other methods

### Use Cases:
- High-quality open-ended generation
- Story/article generation (reduce repetition)
- Long-form text generation
- Dialogue systems (varied responses)
- Content creation
- Academic/technical writing

## Comparison Summary

In [ ]:
import pandas as pd

comparison_data = {
    'Method': [
        'Greedy Search',
        'Beam Search',
        'Top-k Sampling',
        'Top-p Sampling',
        'Temperature',
        'Contrastive'
    ],
    'Speed': [
        'Very Fast',
        'Medium',
        'Fast',
        'Fast',
        'Very Fast',
        'Slow'
    ],
    'Quality': [
        'Poor',
        'Good',
        'Medium',
        'Good',
        'Medium-Good',
        'Excellent'
    ],
    'Diversity': [
        'None',
        'Low',
        'High',
        'High',
        'Controllable',
        'High'
    ],
    'Deterministic': [
        'Yes',
        'Yes',
        'No',
        'No',
        'No',
        'Yes'
    ],
    'Memory': [
        'Low',
        'High',
        'Low',
        'Low',
        'Low',
        'High'
    ],
    'Best For': [
        'Fast baselines',
        'Translation/Summary',
        'Creative text',
        'Creative text',
        'Diversity control',
        'Open-ended generation'
    ]
}

df = pd.DataFrame(comparison_data)
print(df.to_string(index=False))

## Decision Tree: Choosing the Right Decoding Method

```
Start
│
├─ Need deterministic output?
│  ├─ YES → Greedy or Beam Search
│  │  ├─ Is speed critical?
│  │  │  ├─ YES → Greedy Search
│  │  │  └─ NO → Beam Search
│  │
│  └─ NO → Stochastic methods
│
├─ Is quality critical?
│  ├─ YES → Beam Search or Contrastive
│  └─ NO → Greedy or Sampling
│
├─ Repetition a problem?
│  ├─ YES → Contrastive Search or Top-p
│  └─ NO → Any method works
│
└─ Need variety in outputs?
   ├─ YES → Top-p, Top-k, or Contrastive
   └─ NO → Greedy or Beam
```

## Practical Recommendations by Task

In [ ]:
recommendations = {
    'Machine Translation': {
        'Primary': 'Beam Search (num_beams=5-10)',
        'Why': 'High-quality deterministic output needed',
        'Secondary': 'Greedy for fast baselines'
    },
    'Summarization': {
        'Primary': 'Beam Search (num_beams=3-5)',
        'Why': 'Need coherent, deterministic summary',
        'Secondary': 'Top-p (p=0.95) for diverse summaries'
    },
    'Story/Creative Writing': {
        'Primary': 'Top-p Sampling (p=0.9-0.95)',
        'Why': 'Need diverse, creative outputs',
        'Secondary': 'Contrastive (penalty_alpha=0.6) to reduce repetition'
    },
    'Dialogue/Chatbot': {
        'Primary': 'Top-p (p=0.9) + Temperature (0.7-0.9)',
        'Why': 'Natural, varied responses',
        'Secondary': 'Contrastive for longer conversations'
    },
    'Question Answering': {
        'Primary': 'Beam Search (num_beams=3)',
        'Why': 'Factual accuracy needed',
        'Secondary': 'Greedy for fast QA'
    },
    'Code Generation': {
        'Primary': 'Beam Search (num_beams=5)',
        'Why': 'Correctness critical',
        'Secondary': 'Top-p (p=0.95) for diverse solutions'
    },
    'Data Augmentation': {
        'Primary': 'Top-k (k=40-50) or Top-p (p=0.95)',
        'Why': 'Need diverse variations',
        'Secondary': 'Temperature (1.0-1.5) for more variation'
    },
    'Real-time Generation': {
        'Primary': 'Greedy or Top-p (p=0.9)',
        'Why': 'Speed is critical',
        'Secondary': 'Temperature for controlling creativity'
    }
}

for task, config in recommendations.items():
    print(f"\n{'='*60}")
    print(f"Task: {task}")
    print(f"{'='*60}")
    for key, value in config.items():
        print(f"{key:15}: {value}")

## Key Takeaways

1. **Greedy Search**: Fastest but lowest quality. Use only for speed-critical baselines.

2. **Beam Search**: Best for high-quality deterministic tasks (translation, summarization).

3. **Top-k Sampling**: Good for creative text, but may include unlikely tokens.

4. **Top-p (Nucleus) Sampling**: Adaptive and effective for creative generation. Recommended for most sampling tasks.

5. **Temperature Sampling**: Use alongside other methods to control diversity-quality tradeoff.

6. **Contrastive Search**: State-of-the-art for open-ended generation, reduces repetition.

7. **Combination is key**: Modern approaches combine multiple methods:
   - Top-p + Temperature for balanced generation
   - Beam Search with length penalties for summaries
   - Contrastive + Temperature for best diversity and quality

8. **Task matters**: Choose method based on:
   - Quality requirements (deterministic vs diverse)
   - Speed constraints
   - Repetition sensitivity
   - Domain-specific needs